In [1]:
!pip install pandasql; 
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 38.2 MB/s eta 0:00:00a 0:00:01


In [2]:
from template_q import question_templates
import os
import pandas as pd
import pandasql as ps
import duckdb as db
import random

In [11]:
curr_path = '/Users/rishabhbaral/Documents/ASU/Classes/Spring25/CSE576-NLP/Course_Project/SportsT2T/Cricket_Commentary'
table_path = 'Ball_by_Ball_Commentary_Live_Score-WI_SL_2nd_ODI.csv'
table_file = os.path.join(curr_path,table_path)

In [12]:
table_file

'/Users/rishabhbaral/Documents/ASU/Classes/Spring25/CSE576-NLP/Course_Project/SportsT2T/Cricket_Commentary/Ball_by_Ball_Commentary_Live_Score-WI_SL_2nd_ODI.csv'

In [13]:
listdir = os.listdir(curr_path)

In [14]:
df = pd.read_csv(table_file)

In [15]:
df

,raw_data,overs,runs,team_runs,commentary,bowler,batsman,batsman_runs,batsman_fours,batsman_sixes,batsman_bowls_faced,bowler_bowls_done,bowler_runs_given,bowler_wickets,dismissal,runs_given_bool
0,"0.1 \n1 \nPradeep to Lewis, 1 run \nback of a ...",0.1,1,1.0,"Pradeep to Lewis, 1 run .back of a length, wi...",Pradeep,Lewis,1.0,0,0,1,1,1.0,0,NaN,1
1,"0.2 \n1 \nPradeep to Hope, 1 run \nsqueezed of...",0.2,1,1.0,"Pradeep to Hope, 1 run .squeezed off a full l...",Pradeep,Hope,1.0,0,0,1,1,1.0,0,NaN,1
2,"0.3 \n• \nPradeep to Lewis, no run \na back-fo...",0.3,0,0.0,"Pradeep to Lewis, no run .a back-foot slap in...",Pradeep,Lewis,0.0,0,0,1,1,0.0,0,NaN,0
3,"0.4 \n4 \nPradeep to Lewis, FOUR runs \nclonke...",0.4,4,4.0,"Pradeep to Lewis, FOUR runs .clonked through ...",Pradeep,Lewis,4.0,1,0,1,1,4.0,0,NaN,1
4,"0.5 \n• \nPradeep to Lewis, no run \nsliding b...",0.5,0,0.0,"Pradeep to Lewis, no run .sliding back and ac...",Pradeep,Lewis,0.0,0,0,1,1,0.0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,"48.6 \n1 \nChameera to Pooran, 1 run \ntoo ful...",48.6,1,1.0,"Chameera to Pooran, 1 run .too full and strai...",Chameera,Pooran,1.0,0,0,1,1,1.0,0,NaN,1
298,"49.1 \n• \nPradeep to Pooran, no run \na massi...",49.1,0,0.0,"Pradeep to Pooran, no run .a massive wind-up ...",Pradeep,Pooran,0.0,0,0,1,1,0.0,0,NaN,0
299,"49.2 \n4 \nPradeep to Pooran, FOUR runs \nclat...",49.2,4,4.0,"Pradeep to Pooran, FOUR runs .clattered throu...",Pradeep,Pooran,4.0,1,0,1,1,4.0,0,NaN,1
300,"49.3 \n4 \nPradeep to Pooran, FOUR runs \nflas...",49.3,4,4.0,"Pradeep to Pooran, FOUR runs .flashed through...",Pradeep,Pooran,4.0,1,0,1,1,4.0,0,NaN,1


In [16]:
def run_query(template, df, **kwargs):
    # kwargs includes variable names like batsman="Kohli"
    query = template.format(table_name="df", **kwargs)
    print(query)
    return ps.sqldf(query, locals())

query_template = "SELECT overs, bowler FROM {table_name} WHERE batsman = '{batsman}' AND batsman_runs = 4 LIMIT 1;"
result_df = run_query(query_template, df, batsman="Tamim")
print(result_df)

SELECT overs, bowler FROM df WHERE batsman = 'Tamim' AND batsman_runs = 4 LIMIT 1;
Empty DataFrame
Columns: [overs, bowler]
Index: []


In [17]:
for q in question_templates:
    print(q['id'])
    print()
    print(q['question'])
    print()
    print("*"*60)
    print()

1

Find the bowler who has given the most runs and how many runs has the bowler given?

************************************************************

2

Which batsman has faced the most balls and how many balls has the batsman faced?

************************************************************

3

In which over did {batsman} hit their first four and who was the bowler?

************************************************************

4

How many fours did {batsman} hit against {bowler}? Show the tally in table.

************************************************************

5

What is the total number of wickets taken by {bowler1} and {bowler2} combined?

************************************************************

6

Who bowled the most number of balls and how many balls has that bowler bowled?

************************************************************

7

How many runs did {batsman} score?

************************************************************

8

What is the average number o

In [18]:
import random

def sample_params(df, variables, num_samples=5):
    batsmen = df['batsman'].unique().tolist()
    bowlers = df['bowler'].unique().tolist()
    overs = df['overs'].unique().tolist()
    sampled_questions = []

    for _ in range(num_samples):
        params = {}
        params['table_name'] = 'df'

        # Sample batsman(s)
        if 'batsman1' in variables and 'batsman2' in variables:
            batsman1, batsman2 = random.sample(batsmen, 2)
            params['batsman1'] = batsman1
            params['batsman2'] = batsman2
        elif 'batsman' in variables:
            batsman = random.choice(batsmen)
            params['batsman'] = batsman

        # Sample bowler(s)
        if 'bowler' in variables:
            if 'batsman' in params:
                deliveries = df[df['batsman'] == params['batsman']]
                if deliveries.empty:
                    continue
                bowlers_faced = deliveries['bowler'].unique().tolist()
                if not bowlers_faced:
                    continue
                params['bowler'] = random.choice(bowlers_faced)
            else:
                params['bowler'] = random.choice(bowlers)

        if 'bowler1' in variables and 'bowler2' in variables:
            params['bowler1'], params['bowler2'] = random.sample(bowlers, 2)

        # Sample over ranges
        if 'x1' in variables and 'x2' in variables:
            if 'batsman' in params and 'bowler' in params:
                relevant_overs = df[(df['batsman'] == params['batsman']) & (df['bowler'] == params['bowler'])]['overs'].tolist()
                if len(relevant_overs) < 2:
                    continue  # skip this sample if not enough overs to pick a range
                x1 = min(relevant_overs)
                x2 = max(relevant_overs)
                params['x1'] = x1
                params['x2'] = x2
            else:
                if len(overs) < 2:
                    continue
                x1, x2 = sorted(random.sample(overs, 2))
                params['x1'] = x1
                params['x2'] = x2

        sampled_questions.append(params)

    return sampled_questions



# Run query with params
def run_query(template, df, param_values):
    question_text = template["question"].format(**param_values)
    query_text = template["query"].format(**param_values)
    result_df = db.query_df(df, param_values["table_name"], query_text).to_df()

    return question_text, query_text, result_df


# Run multiple questions from one template
def run_multiple_questions(template, df, num_samples=5):
    variables = template["variables"]
    if len(variables)>1:
        param_sets = sample_params(df, variables ,num_samples)
    else:
        param_sets = sample_params(df, variables ,1)
    results = []
    for params in param_sets:
        try:
            question, query, result_df = run_query(template, df, params)
            results.append({
                "question": question,
                "query": query,
                "params": params,
                "result": result_df
            })
        except Exception as e:
            results.append({
                "question": template["question"],
                "query": template["query"],
                "error": str(e),
                "params": params
            })
    return results


# Run for 5 different random params
c = 1
for ques in question_templates:
    results = run_multiple_questions(ques, df, num_samples=1)
    for res in results:
        print("Qno: ",c)
        print(res['question'])
        print()
        print(res['query'])
        print(res['params'])
        if 'result' in res:
            print(res['result'])
        else:
            print("Error")
            print(res['error'])
        print("*"*60)
        print()
        c = c+1

# Display results
"""for res in results:
    print(f"\n📝 {res['question']}")
    print(f"→ Query: {res['query']}")
    print(f"→ Params: {res['params']}")
    if "result" in res:
        print(res["result"])
    else:
        print("Error:", res["error"])"""

Qno:  1
Find the bowler who has given the most runs and how many runs has the bowler given?

SELECT bowler, SUM(bowler_runs_given) AS total_runs FROM df GROUP BY bowler ORDER BY total_runs DESC LIMIT 1;
{'table_name': 'df'}
    bowler  total_runs
0  Pradeep        66.0
************************************************************

Qno:  2
Which batsman has faced the most balls and how many balls has the batsman faced?

SELECT batsman, SUM(batsman_bowls_faced) AS total_balls FROM df GROUP BY batsman ORDER BY total_balls DESC LIMIT 1;
{'table_name': 'df'}
  batsman  total_balls
0   Lewis        121.0
************************************************************

Qno:  3
In which over did Pollard hit their first four and who was the bowler?

SELECT overs, bowler FROM df WHERE batsman = 'Pollard' AND batsman_runs = 4 ORDER BY overs ASC LIMIT 1;
{'table_name': 'df', 'batsman': 'Pollard'}
   overs     bowler
0   44.2  Hasaranga
************************************************************

Qno:

'for res in results:\n    print(f"\n📝 {res[\'question\']}")\n    print(f"→ Query: {res[\'query\']}")\n    print(f"→ Params: {res[\'params\']}")\n    if "result" in res:\n        print(res["result"])\n    else:\n        print("Error:", res["error"])'

In [9]:
## make till 125
## manually check each question (wording)
## gold sql review
## 5 or 10 matches, see the pattern and find the strategy to initialize
## 3 different entries for a variables like batsman, bowler
## run model on mini-set first (3-5 matches)
## every template to get 6

In [19]:
df[(df['runs']=='1nb')]

,raw_data,overs,runs,team_runs,commentary,bowler,batsman,batsman_runs,batsman_fours,batsman_sixes,batsman_bowls_faced,bowler_bowls_done,bowler_runs_given,bowler_wickets,dismissal,runs_given_bool
187,"31.2 \n1nb \nChameera to Lewis, (no ball) \nat...",31.2,1nb,1.0,"Chameera to Lewis, (no ball) .attempted pick-...",Chameera,Lewis,0.0,0,0,1,0,1.0,0,NaN,1


In [20]:
44*5

220

In [21]:
[
    {
        "id": 44,
        "question": "Calculate the average runs scored per ball by {batsman1} and {batsman2} against bowlers who bowled at least 5 overs between overs {x1} and {x2}",
        "query": (
                "SELECT AVG(batsman_runs * 1.0 / NULLIF(batsman_bowls_faced, 0)) AS average_runs_per_ball "
                "FROM {table_name} "
                "WHERE batsman IN ('{batsman1}', '{batsman2}') "
                "AND bowler_bowls_done >= 30 "
                "AND overs BETWEEN {x1} AND {x2};"
            ),
        "variables": ["table_name", "batsman1", "batsman2", "x1", "x2"]
    },
    {
        "id": 40,
        "question": "Calculate the average runs scored per ball by {batsman1} and {batsman2} against bowlers who bowled at least 5 overs between overs {x1} and {x2}",
        "query": "SELECT AVG(CAST(batsman_runs AS FLOAT) / batsman_bowls_faced) AS average_runs_per_ball FROM {table_name} WHERE batsman IN ('{batsman1}', '{batsman2}') AND bowler_bowls_done >= 30 AND overs BETWEEN {x1} AND {x2};",
        "variables": ["table_name", "batsman1", "batsman2", "x1", "x2"]
    }
]

[{'id': 44,
  'question': 'Calculate the average runs scored per ball by {batsman1} and {batsman2} against bowlers who bowled at least 5 overs between overs {x1} and {x2}',
  'query': "SELECT AVG(batsman_runs * 1.0 / NULLIF(batsman_bowls_faced, 0)) AS average_runs_per_ball FROM {table_name} WHERE batsman IN ('{batsman1}', '{batsman2}') AND bowler_bowls_done >= 30 AND overs BETWEEN {x1} AND {x2};",
  'variables': ['table_name', 'batsman1', 'batsman2', 'x1', 'x2']},
 {'id': 40,
  'question': 'Calculate the average runs scored per ball by {batsman1} and {batsman2} against bowlers who bowled at least 5 overs between overs {x1} and {x2}',
  'query': "SELECT AVG(CAST(batsman_runs AS FLOAT) / batsman_bowls_faced) AS average_runs_per_ball FROM {table_name} WHERE batsman IN ('{batsman1}', '{batsman2}') AND bowler_bowls_done >= 30 AND overs BETWEEN {x1} AND {x2};",
  'variables': ['table_name', 'batsman1', 'batsman2', 'x1', 'x2']}]